<a href="https://colab.research.google.com/github/Martinz77-max/Martinz77-max/blob/main/avatarify.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Avatarify Colab Server

This Colab notebook is for running Avatarify rendering server. It allows you to run Avatarify on your computer **without GPU** in this way:

1. When this notebook is executed, it starts listening for incoming requests from your computer;
1. You start the client on your computer and it connects to the notebook and starts sending requests;
1. This notebooks receives the requests from your computer, renders avatar images and sends them back;

To this end, all the heavy work is offloaded from your computer to this notebook so you don't need to have a beafy hardware on your PC anymore.


## Start the server
Run the cells below (Shift+Enter) sequentially and pay attention to the hints and instructions included in this notebook.

At the end you will get a command for running the client on your computer.

## Start the client

Make sure you have installed the latest version of Avatarify on your computer. Refer to the [README](https://github.com/alievk/avatarify#install) for the instructions.

When it's ready execute this notebook and get the command for running the client on your computer.


### Technical details

The client on your computer connects to the server via `ngrok` TCP tunnel or a reverse `ssh` tunnel.

`ngrok`, while easy to use, can induce a considerable network lag ranging from dozens of milliseconds to a second. This can lead to a poor experience.

A more stable connection could be established using a reverse `ssh` tunnel to a host with a public IP, like an AWS `t3.micro` (free) instance. This notebook provides a script for creating a tunnel, but launching an instance in a cloud is on your own (find the manual below).

# Install

### Avatarify
Follow the steps below to clone Avatarify and install the dependencies.

In [77]:
!cd /content
!rm -rf *

In [78]:
!git clone https://github.com/alievk/avatarify.git

Cloning into 'avatarify'...
remote: Enumerating objects: 1514, done.
remote: Total 1514 (delta 0), reused 0 (delta 0), pack-reused 1514 (from 1)
Receiving objects: 100% (1514/1514), 5.69 MiB | 22.84 MiB/s, done.
Resolving deltas: 100% (967/967), done.


In [10]:
cd avatarify

/content/avatarify/avatarify/avatarify


In [12]:
!git clone https://github.com/alievk/first-order-model.git fomm
!sudo apt-get install -y build-essential libyaml-dev
!pip install face-alignment==1.0.0 msgpack_numpy pyyaml

fatal: destination path 'fomm' already exists and is not an empty directory.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
build-essential is already the newest version (12.9ubuntu3).
libyaml-dev is already the newest version (0.2.2-1build2).
0 upgraded, 0 newly installed, 0 to remove and 53 not upgraded.


In [13]:
!./scripts/download_data.sh

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  228M  100  228M    0     0   145M      0  0:00:01  0:00:01 --:--:--  145M
Expected checksum: 8a45a24037871c045fbb8a6a8aa95ebc
Found checksum:    8a45a24037871c045fbb8a6a8aa95ebc  vox-adv-cpk.pth.tar


### ngrok
Follow the steps below to setup ngrok. You will also need to sign up on the ngrok site and get your authtoken (free).


In [14]:
# Download ngrok
!scripts/get_ngrok.sh

ngrok is not found, installing...
Done!


# Run
Start here if the runtime was restarted after installation.

In [15]:
cd /content/avatarify

/content/avatarify


In [16]:
#!git pull origin

In [39]:
from subprocess import Popen, PIPE
import shlex
import json
import time

def run_with_pipe(command):
  commands = list(map(shlex.split,command.split("|")))
  ps = Popen(commands[0], stdout=PIPE, stderr=PIPE)
  for command in commands[1:]:
    ps = Popen(command, stdin=ps.stdout, stdout=PIPE, stderr=PIPE)
  return ps.stdout.readlines()


def get_tunnel_adresses():
  max_retries = 10
  retry_delay_seconds = 2
  info = None
  for i in range(max_retries):
    try:
      # Add a timeout for curl to prevent hanging indefinitely
      raw_info = run_with_pipe("curl --max-time 5 http://localhost:4040/api/tunnels")
      if raw_info:
        info = json.loads(raw_info[0])
        # Check if tunnels list is not empty or if ngrok is actually serving
        if 'tunnels' in info and len(info['tunnels']) > 0:
          break  # Success, break out of retry loop
    except json.JSONDecodeError:
      print(f"Attempt {i+1}/{max_retries}: ngrok API not ready (JSON decode error), retrying in {retry_delay_seconds} seconds...")
      time.sleep(retry_delay_seconds)
    except Exception as e:
      # Catch other potential errors, e.g., curl failing before JSON is returned
      print(f"Attempt {i+1}/{max_retries}: Error accessing ngrok API: {e}, retrying in {retry_delay_seconds} seconds...")
      time.sleep(retry_delay_seconds)

  if not info or 'tunnels' not in info or len(info['tunnels']) == 0:
    raise Exception("Failed to get ngrok tunnel information after multiple retries or no tunnels found.")

  in_addr = None
  out_addr = None
  for tunnel in info['tunnels']:
    url = tunnel['public_url']
    port = url.split(':')[-1]
    local_port = tunnel['config']['addr'].split(':')[-1]
    print(f'{url} -> {local_port} [{tunnel["name"]}]')
    if tunnel['name'] == 'input':
      in_addr = url
    elif tunnel['name'] == 'output':
      out_addr = url
    else:
      print(f'unknown tunnel: {tunnel["name"]}')

  if in_addr is None or out_addr is None:
    raise Exception("Could not find both input and output tunnel addresses.")

  return in_addr, out_addr

In [18]:
# Input and output ports for communication
local_in_port = 5557
local_out_port = 5558

# Start the worker


In [20]:
print('Listing contents of current directory:')
!ls -F

Listing contents of current directory:
avatarify/


In [27]:
# (Re)Start the worker
with open('/tmp/run.txt', 'w') as f:
  ps = Popen(
      shlex.split(f'./run.sh --is-worker --in-port {local_in_port} --out-port {local_out_port} --no-vcam --no-conda'),
      stdout=f, stderr=f, cwd='/content/avatarify/avatarify/avatarify')
  time.sleep(3)

In [28]:
# Check if the worker started
!ps aux | grep 'python3 afy/cam_fomm.py' | grep -v grep | tee /tmp/ps_run
!if [[ $(cat /tmp/ps_run | wc -l) == "0" ]]; then echo "Worker failed to start"; cat /tmp/run.txt; else echo "Worker started"; fi

root       20686 16.2  4.5 5463564 598200 ?      S    08:45   0:02 python3 afy/cam_fomm.py --config fomm/config/vox-adv-256.yaml --checkpoint vox-adv-cpk.pth.tar --virt-cam 9 --relative --adapt_scale --is-worker --in-port 5557 --out-port 5558 --no-stream
root       20700  0.0  2.6 5529100 346056 ?      Sl   08:45   0:00 python3 afy/cam_fomm.py --config fomm/config/vox-adv-256.yaml --checkpoint vox-adv-cpk.pth.tar --virt-cam 9 --relative --adapt_scale --is-worker --in-port 5557 --out-port 5558 --no-stream
root       20703  0.0  2.5 5463564 343640 ?      S    08:45   0:00 python3 afy/cam_fomm.py --config fomm/config/vox-adv-256.yaml --checkpoint vox-adv-cpk.pth.tar --virt-cam 9 --relative --adapt_scale --is-worker --in-port 5557 --out-port 5558 --no-stream
root       20704  0.0  2.6 5529100 346092 ?      Sl   08:45   0:00 python3 afy/cam_fomm.py --config fomm/config/vox-adv-256.yaml --checkpoint vox-adv-cpk.pth.tar --virt-cam 9 --relative --adapt_scale --is-worker --in-port 5557 --out-po

In [26]:
print('Searching for run.sh:')
!find /content -name run.sh

Searching for run.sh:
/content/avatarify/avatarify/avatarify/run.sh


In [24]:
print('Listing contents of /content/avatarify:')
!ls -F /content/avatarify

Listing contents of /content/avatarify:
avatarify/


This command should print lines if the worker is successfully started

In [29]:
!ps aux | grep 'python3 afy/cam_fomm.py' | grep -v grep | tee /tmp/ps_run
!if [[ $(cat /tmp/ps_run | wc -l) == "0" ]]; then echo "Worker failed to start"; cat /tmp/run.txt; else echo "Worker started"; fi

root       20686  5.9  4.5 5463564 598200 ?      S    08:45   0:02 python3 afy/cam_fomm.py --config fomm/config/vox-adv-256.yaml --checkpoint vox-adv-cpk.pth.tar --virt-cam 9 --relative --adapt_scale --is-worker --in-port 5557 --out-port 5558 --no-stream
root       20700  0.0  2.6 5529100 346060 ?      Sl   08:45   0:00 python3 afy/cam_fomm.py --config fomm/config/vox-adv-256.yaml --checkpoint vox-adv-cpk.pth.tar --virt-cam 9 --relative --adapt_scale --is-worker --in-port 5557 --out-port 5558 --no-stream
root       20703  0.0  2.5 5463564 343640 ?      S    08:45   0:00 python3 afy/cam_fomm.py --config fomm/config/vox-adv-256.yaml --checkpoint vox-adv-cpk.pth.tar --virt-cam 9 --relative --adapt_scale --is-worker --in-port 5557 --out-port 5558 --no-stream
root       20704  0.0  2.6 5529100 346092 ?      Sl   08:45   0:00 python3 afy/cam_fomm.py --config fomm/config/vox-adv-256.yaml --checkpoint vox-adv-cpk.pth.tar --virt-cam 9 --relative --adapt_scale --is-worker --in-port 5557 --out-po

# Open ngrok tunnel

#### Get ngrok token
Go to https://dashboard.ngrok.com/auth/your-authtoken (sign up if required), copy your authtoken and put it below.

In [30]:
# Paste your authtoken here in quotes
authtoken = "3FfGOW9fQukubkgSyvca5uVEvwg_22NDDPAkwPykwjPQ1hLGL"

Set your region

Code | Region
--- | ---
us | United States
eu | Europe
ap | Asia/Pacific
au | Australia
sa | South America
jp | Japan
in | India

In [31]:
# Set your region here in quotes
region = "us"

In [32]:
config =\
f"""
version: 2
authtoken: {authtoken}
region: {region}
console_ui: False
tunnels:
  input:
    addr: {local_in_port}
    proto: tcp
  output:
    addr: {local_out_port}
    proto: tcp
"""

with open('ngrok.conf', 'w') as f:
  f.write(config)

In [42]:
# This cell is no longer needed as its content has been merged into JAyPH2t2C64H.
# It can be safely ignored or removed.
# ps = Popen('./avatarify/avatarify/scripts/open_tunnel_ngrok.sh', stdout=PIPE, stderr=PIPE)
# time.sleep(3)

In [52]:
import time
from subprocess import Popen, PIPE
import shlex
import os # Import os module to handle paths

# Open the ngrok tunnel
print("Opening ngrok tunnel...")

# Modify the script content and then run it directly
# script_path is relative to the current Python cwd, which is /content/avatarify
script_path = './avatarify/avatarify/scripts/open_tunnel_ngrok.sh'
with open(script_path, 'r') as f:
  script_content = f.read()

# Apply fixes to the script content
# 1. Correct ngrok path and ngrok.conf path relative to script_dir
#    The original line from !cat b12dc052 was: 'cmd="../../../ngrok start --all --config ngrok.conf"'
#    It needs to be: 'cmd="../ngrok start --all --config ../../../ngrok.conf"'
script_content = script_content.replace(
    'cmd="../../../ngrok start --all --config ngrok.conf"',
    'cmd="../ngrok start --all --config ../../../ngrok.conf"'
)
# 2. Quote $cmd in grep command (this was already correctly handled in the original script and previous fix, so this line can be removed or kept as it's idempotent now)
#    Keeping it for robustness in case the script's original content changes again.
script_content = script_content.replace('grep $cmd', 'grep "$cmd"')

# Write the modified content back to the script
with open(script_path, 'w') as f:
  f.write(script_content)

# Make sure the script is executable
!chmod +x {script_path}

# Set the working directory for the shell script to its own directory
# This ensures that relative paths within the script (like ../ngrok) are resolved correctly.
script_dir = os.path.dirname(script_path)

# The Popen command should execute the script relative to its own directory
ps = Popen(['./' + os.path.basename(script_path)], stdout=PIPE, stderr=PIPE, cwd=script_dir)
time.sleep(5) # Give ngrok more time to start

# Get tunnel addresses
try:
  in_addr, out_addr = get_tunnel_adresses()
  print("Tunnel opened successfully!")
  print(f"Input Tunnel Address: {in_addr}")
  print(f"Output Tunnel Address: {out_addr}")
except Exception as e:
  print(f"Error getting tunnel addresses: {e}")
  print("Ngrok script output (if any):")
  # Print both stdout and stderr from the ngrok script
  stdout_lines = ps.stdout.readlines()
  stderr_lines = ps.stderr.readlines()
  if stdout_lines:
    for line in stdout_lines:
      print(line.decode().strip())
  if stderr_lines:
    for line in stderr_lines:
      print(line.decode().strip())
  if not stdout_lines and not stderr_lines:
    print("No output from ngrok script.")
  print("Please check the ngrok setup and try again.")

Opening ngrok tunnel...
Error getting tunnel addresses: Failed to get ngrok tunnel information after multiple retries or no tunnels found.
Ngrok script output (if any):
Opening tunnel
ERROR:  Error reading configuration file 'ngrok.conf': open /content/avatarify/avatarify/avatarify/scripts/ngrok.conf: no such file or directory
Please check the ngrok setup and try again.


In [47]:
!cat /content/avatarify/avatarify/avatarify/scripts/open_tunnel_ngrok.sh

#!/usr/bin/env bash

cmd="../../../ngrok start --all --config ngrok.conf"

kill -9 $(ps aux | grep "$cmd" | awk '{print $2}') 2> /dev/null

echo Opening tunnel
$cmd



In [48]:
!find /content -name ngrok

/content/avatarify/avatarify/avatarify/ngrok


### [Optional] AWS proxy
Alternatively you can create a ssh reverse tunnel to an AWS `t3.micro` instance (it's free). It has lower latency than ngrok.

1. In your AWS console go to Services -> EC2 -> Instances -> Launch Instance;
1. Choose `Ubuntu Server 18.04 LTS` AMI;
1. Choose `t3.micro` instance type and press Review and launch;
1. Confirm your key pair and press Launch instances;
1. Go to the security group of this instance and edit inbound rules. Add TCP ports 5557 and 5558 and set Source to Anywhere. Press Save rules;
1. ssh into the instance (you can find the command in the Instances if you click on the Connect button) and add this line in the end of `/etc/ssh/sshd_config`:
```
GatewayPorts yes
```
then restart `sshd`
```
sudo service sshd restart
```
1. Copy your `key_pair.pem` by dragging and dropping it into avatarify folder in this notebook;
1. Use the command below to open the tunnel;
1. Start client with a command (substitute `run_mac.sh` with `run_windows.bat` or `run.sh`)
```
./run_mac.sh --is-client --in-addr tcp://instace.compute.amazonaws.com:5557 --out-addr tcp://instance.compute.amazonaws.com:5558
```

In [ ]:
# Open reverse ssh tunnel (uncomment line below)
# !./scripts/open_tunnel_ssh.sh key_pair.pem ubuntu@instance.compute.amazonaws.com

# Start the client
When you run the cell below it will print a command. Run this command on your computer:

1. Open a terminal (in Windows open `Anaconda Prompt`);
2. Change working directory to the `avatarify` directory:</br>
* Windows (change `C:\path\to\avatarify` to your path)</br>
`cd C:\path\to\avatarify`</br></br>
* Mac/Linux (change `/path/to/avatarify` to your path)</br>
`cd /path/to/avatarify`
3. Copy-paste to the terminal the command below and run;
4. It can take some time to connect (usually up to 10 seconds). If the preview window doesn't appear in a minute or two, look for the errors above in this notebook and report in the [issues](https://github.com/alievk/avatarify/issues) or [Slack](https://join.slack.com/t/avatarify/shared_invite/zt-dyoqy8tc-~4U2ObQ6WoxuwSaWKKVOgg).

In [ ]:
print('Copy-paste to the terminal the command below and run (press Enter)\n')
print('Mac:')
print(f'./run_mac.sh --is-client --in-addr {in_addr} --out-addr {out_addr}')
print('\nWindows:')
print(f'run_windows.bat --is-client --in-addr {in_addr} --out-addr {out_addr}')
print('\nLinux:')
print(f'./run.sh --is-client --in-addr {in_addr} --out-addr {out_addr}')

# Logs

If something doesn't work as expected, please run the cells below and include the logs in your report.

In [ ]:
#@title
!cat ./var/log/cam_fomm.log | head -100

In [ ]:
#@title
!cat ./var/log/recv_worker.log | tail -100

In [ ]:
#@title
!cat ./var/log/predictor_worker.log | tail -100

In [ ]:
#@title
!cat ./var/log/send_worker.log | tail -100